# Multi-Algo Comparison Walkthrough

**Run multiple trading strategies side-by-side and compare their performance**

---

This notebook demonstrates `SharedReplayEngine`, which runs multiple trading algorithms
against the same historical data in a single pass. Market data is loaded once and shared;
each algorithm gets its own isolated `TradingContext` with independent positions,
orders, and equity tracking.

By the end you will:
- Understand when to use `SharedReplayEngine` vs `BacktestEngine`
- Know how to define multiple strategies and run them in one pass
- Be able to compare strategies by PnL, Sharpe ratio, profit factor, and win rate
- Know how to export a comparison as an HTML report or JSON file

**Estimated reading time: 15–20 minutes**

## Why shared replay?

When comparing strategies, the naive approach is to run `BacktestEngine` separately
for each strategy. This works, but:

1. **Data is loaded N times** — each `BacktestEngine` reads the same Parquet files
   independently. For IDC data (~1.5 GB/zone/year) with 8 strategies, that is 12 GB
   of peak memory.

2. **Results are not directly comparable** — separate runs may encounter edge cases
   at different times if the data loading is not exactly reproducible.

`SharedReplayEngine` solves both:
- DA data is loaded **once** and shared as a single Pandas DataFrame
- IDC data uses a single `SlidingWindow` advanced in lockstep
- All algorithms process timestamp T before any moves to T+1
- Each algorithm has completely isolated positions, orders, and fills

## Setup

Make sure you have generated the fixture data first:

In [1]:
import subprocess
import sys
from pathlib import Path

# Generate fixtures if not already present
fixtures_dir = Path('../tests/fixtures')
if not (fixtures_dir / 'da_prices.parquet').exists():
    subprocess.run([sys.executable, str(fixtures_dir / 'generate.py')], check=True)
    print('Fixtures generated.')
else:
    print('Fixtures already present.')

Fixtures already present.


In [2]:
from __future__ import annotations

from datetime import date
from decimal import Decimal
from pathlib import Path

from nexa_backtest.algo import SimpleAlgo
from nexa_backtest.context import TradingContext
from nexa_backtest.engines.shared import SharedReplayEngine
from nexa_backtest.exceptions import SignalError
from nexa_backtest.types import AuctionInfo, Order

DATA_DIR = Path('../tests/fixtures')

## Define the strategies

We will compare three variations of the same forecast-based strategy, differing
only in how aggressively they trade:

| Strategy | Threshold | Expected behaviour |
|---|---|---|
| **Conservative** | 8 EUR/MWh | Fewest fills, highest average alpha per fill |
| **Moderate** | 5 EUR/MWh | Balanced — more fills, slightly lower alpha |
| **Aggressive** | 0.1 EUR/MWh | Most fills, lowest average alpha per fill |

Each strategy buys when `forecast - clearing_price > threshold`, bidding at
`forecast - threshold` (a conservative bid that still fills).

In [3]:
class ConservativeAlgo(SimpleAlgo):
    """Buy when forecast is at least 8 EUR/MWh above clearing price."""

    def on_setup(self, ctx: TradingContext) -> None:
        self.subscribe_signal('price_forecast')
        self.threshold = Decimal('8.0')

    def on_auction_open(self, ctx: TradingContext, auction: AuctionInfo) -> None:
        try:
            signal = ctx.get_signal('price_forecast')
        except SignalError:
            return
        ctx.place_order(Order.buy(
            product_id=auction.product_id,
            volume_mw=Decimal('10'),
            price_eur_mwh=Decimal(str(signal.value)) - self.threshold,
        ))


class ModerateAlgo(SimpleAlgo):
    """Buy when forecast is at least 5 EUR/MWh above clearing price."""

    def on_setup(self, ctx: TradingContext) -> None:
        self.subscribe_signal('price_forecast')
        self.threshold = Decimal('5.0')

    def on_auction_open(self, ctx: TradingContext, auction: AuctionInfo) -> None:
        try:
            signal = ctx.get_signal('price_forecast')
        except SignalError:
            return
        ctx.place_order(Order.buy(
            product_id=auction.product_id,
            volume_mw=Decimal('10'),
            price_eur_mwh=Decimal(str(signal.value)) - self.threshold,
        ))


class AggressiveAlgo(SimpleAlgo):
    """Buy whenever forecast is above clearing price (minimal threshold)."""

    def on_setup(self, ctx: TradingContext) -> None:
        self.subscribe_signal('price_forecast')
        self.threshold = Decimal('0.1')

    def on_auction_open(self, ctx: TradingContext, auction: AuctionInfo) -> None:
        try:
            signal = ctx.get_signal('price_forecast')
        except SignalError:
            return
        ctx.place_order(Order.buy(
            product_id=auction.product_id,
            volume_mw=Decimal('10'),
            price_eur_mwh=Decimal(str(signal.value)) - self.threshold,
        ))


print('Strategies defined: ConservativeAlgo, ModerateAlgo, AggressiveAlgo')

Strategies defined: ConservativeAlgo, ModerateAlgo, AggressiveAlgo


## Run the shared replay

Pass all three strategies to `SharedReplayEngine` as a dictionary. The keys
become the display names in the comparison report.

In [4]:
engine = SharedReplayEngine(
    algos={
        'conservative': ConservativeAlgo(),
        'moderate': ModerateAlgo(),
        'aggressive': AggressiveAlgo(),
    },
    exchange='nordpool',
    start=date(2026, 3, 1),
    end=date(2026, 3, 31),
    products=['NO1_DA'],
    data_dir=DATA_DIR,
    initial_capital=Decimal('100000'),
)

comparison = engine.run()
print('Run complete.')

Signal 'price_forecast' has no publication_offset. Values are available at their timestamp. This is correct for actuals but may introduce look-ahead bias for forecasts — consider whether your data is a forecast.


Run complete.


## Side-by-side summary

`comparison.summary()` prints a formatted table with all strategies side-by-side.
It also reports peak memory usage and the estimated saving vs separate backtests.

In [5]:
print(comparison.summary())

  Comparison Results: 2026-03-01 to 2026-03-31 (31 days)
  Exchange: nordpool | Products: NO1_DA

                          conservative          moderate        aggressive
  Total PnL            +106,857.05 EUR   +103,261.12 EUR    +84,388.84 EUR
  vs VWAP                44.79 EUR/MWh     44.79 EUR/MWh     44.79 EUR/MWh
  Sharpe                         47.63             47.19             34.01
  Max Drawdown               -0.00 EUR         -0.00 EUR         -0.00 EUR
  Profit Factor                  14.51              7.64              3.19
  Win Rate                       82.6%             75.6%             64.9%
  Trades                          1480              1733              2168

  Best by PnL: conservative (+106,857.05 EUR)
  Best risk-adjusted: conservative (Sharpe 47.63)

  Memory: 0 MB (saved ~0 MB vs separate backtests)


## Ranking by different metrics

Use `comparison.ranking(metric)` to rank strategies by any metric:
- `'total_pnl'` — total alpha in EUR
- `'sharpe_ratio'` — risk-adjusted returns
- `'profit_factor'` — gross profits / gross losses
- `'win_rate'` — fraction of trades with positive alpha
- `'max_drawdown'` — less negative is better
- `'trades'` — total number of fills

In [6]:
for metric in ['total_pnl', 'sharpe_ratio', 'profit_factor', 'win_rate', 'trades']:
    ranked = comparison.ranking(metric)
    print(f'{metric:<15}: {" > ".join(ranked)}')

total_pnl      : conservative > moderate > aggressive
sharpe_ratio   : conservative > moderate > aggressive
profit_factor  : conservative > moderate > aggressive
win_rate       : conservative > moderate > aggressive
trades         : aggressive > moderate > conservative


## Inspecting individual results

`comparison.results` is a dict mapping display name to `BacktestResult`.
Each `BacktestResult` has the full fill list, PnL breakdown, and metrics.

In [7]:
for name, result in comparison.results.items():
    pnl = result.pnl.total_alpha_eur
    trades = len(result.fills)
    sharpe = float(result.sharpe_ratio) if result.sharpe_ratio else 0.0
    print(f'{name:<15}  PnL: {float(pnl):+,.2f} EUR  Trades: {trades:4d}  Sharpe: {sharpe:.2f}')

conservative     PnL: +106,857.05 EUR  Trades: 1480  Sharpe: 47.63
moderate         PnL: +103,261.12 EUR  Trades: 1733  Sharpe: 47.19
aggressive       PnL: +84,388.84 EUR  Trades: 2168  Sharpe: 34.01


## Best and worst strategies

`comparison.best` and `comparison.worst` return `(name, BacktestResult)` tuples
for the strategies with the highest and lowest total PnL.

In [8]:
best_name, best_result = comparison.best
worst_name, worst_result = comparison.worst

print(f'Best:  {best_name}  ({float(best_result.pnl.total_alpha_eur):+,.2f} EUR)')
print(f'Worst: {worst_name}  ({float(worst_result.pnl.total_alpha_eur):+,.2f} EUR)')
print()
diff = best_result.pnl.total_alpha_eur - worst_result.pnl.total_alpha_eur
print(f'PnL difference: {float(diff):+,.2f} EUR')

Best:  conservative  (+106,857.05 EUR)
Worst: aggressive  (+84,388.84 EUR)

PnL difference: +22,468.21 EUR


## Exporting the comparison report

Export to HTML (self-contained, includes interactive Plotly charts) or JSON
(machine-readable, suitable for CI checks or further analysis).

In [9]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    html_path = Path(tmpdir) / 'comparison.html'
    json_path = Path(tmpdir) / 'comparison.json'

    comparison.to_html(str(html_path))
    comparison.to_json(str(json_path))

    html_size = html_path.stat().st_size
    json_size = json_path.stat().st_size

    print(f'HTML report: {html_size:,} bytes')
    print(f'JSON export: {json_size:,} bytes')
    print()

    payload = json.loads(json_path.read_text())
    print('JSON top-level keys:', list(payload.keys()))
    print('Algo keys:', list(payload['algos'].keys()))
    for name, data in payload['algos'].items():
        print(f'  {name}: PnL={data["total_pnl_eur"]:+,.2f} EUR  trades={data["trade_count"]}')

HTML report: 30,240 bytes
JSON export: 1,374 bytes

JSON top-level keys: ['start_date', 'end_date', 'exchange', 'products', 'peak_memory_bytes', 'estimated_separate_memory_bytes', 'algos']
Algo keys: ['conservative', 'moderate', 'aggressive']
  conservative: PnL=+106,857.05 EUR  trades=1480
  moderate: PnL=+103,261.12 EUR  trades=1733
  aggressive: PnL=+84,388.84 EUR  trades=2168


## Running comparisons from the CLI

The `nexa compare` command accepts multiple `name:path` algo specs:

```bash
nexa compare \
    conservative:examples/multi_algo_comparison.py \
    aggressive:examples/multi_algo_comparison.py \
    --exchange nordpool \
    --start 2026-03-01 \
    --end 2026-03-31 \
    --products NO1_DA \
    --data-dir tests/fixtures \
    --output /tmp/comparison.html
```

If no display name is given, the filename stem is used.

**Maximum: 8 algos per comparison.**  More than that makes the report unreadable.

## Memory efficiency

`comparison.peak_memory_bytes` reports the peak memory used by the shared data layer.
`comparison.estimated_separate_memory_bytes` estimates the cost of running each strategy
separately with its own data copy.

For DA data the saving is small (DA data is tiny). For IDC data across many strategies
the saving can be 100+ GB.

In [10]:
peak_mb = comparison.peak_memory_bytes / (1024 * 1024)
separate_mb = comparison.estimated_separate_memory_bytes / (1024 * 1024)
saved_mb = separate_mb - peak_mb

n = len(comparison.results)
print(f'Peak shared memory:        {peak_mb:.1f} MB')
print(f'Estimated separate memory: {separate_mb:.1f} MB')
print(f'Memory saved:              {saved_mb:.1f} MB')
print(f'  ({n} algos x {peak_mb:.1f} MB = {separate_mb:.1f} MB vs {peak_mb:.1f} MB shared)')

Peak shared memory:        0.2 MB
Estimated separate memory: 0.5 MB
Memory saved:              0.3 MB
  (3 algos x 0.2 MB = 0.5 MB vs 0.2 MB shared)


## Summary

You have learned how to:

1. Define multiple `SimpleAlgo` strategies
2. Run them in lockstep with `SharedReplayEngine`
3. Read `ComparisonResult.summary()` for a side-by-side view
4. Rank strategies with `ranking(metric)`
5. Access per-strategy `BacktestResult` objects via `comparison.results`
6. Export to HTML and JSON with `to_html()` and `to_json()`
7. Understand the memory efficiency of shared replay

**Next steps:**
- Add IDC products (`NO1-QH`) to see the IDC shared replay in action
- Try the `@algo` decorator with async event streams (see notebook 04)
- Combine shared replay with signals and ML models for a full strategy comparison